# 02 . Tokenise every subject and build session features

Runs locally. Produces the two things the model needs, then you move them to Colab,
because `mamba_ssm` needs CUDA and will not build on Apple Silicon.

Outputs

```
tokens_schaefer100.npy    (161, 15000, 100) uint8
session_feats.npy         (161, 100, 292)   float32
labels_mamba.npy          (161,)            int64
subject_ids_mamba.npy     (161,)
```

Session features are per-parcel token unigram and bigram statistics put through the frozen
transform MEG-Mamba ships, which is their documented path for applying the released model
to a new recording without training.

In [10]:
import os, sys, time
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import numpy as np, h5py, scipy.signal, torch

DATA   = '../data'
OUT    = '../results_mamba'
PARCEL = '../results_parcel'
MAMBA  = '/Users/faenegoro/Documents/Neuroimaging/MEG_Mamba/MEG-Mamba'
sys.path.insert(0, '/Users/faenegoro/Documents/Neuroimaging/EphysTokenizer')
sys.path.insert(0, MAMBA)

FS_ORIG, FS_TARGET = 600, 250
W    = np.load(f'{OUT}/schaefer_W_counts.npy')
keep = np.load(f'{PARCEL}/parcel_keep_idx_cortical.npy')
print(f'W {W.shape}  keep_idx {keep.shape}  resample {FS_ORIG} -> {FS_TARGET} Hz')

W (210, 100)  keep_idx (210,)  resample 600 -> 250 Hz


In [11]:
def pca_merge(sig, W):
    """(210, T) -> (100, T) via each parcel's first principal component, sign invariant."""
    out = np.zeros((W.shape[1], sig.shape[1]), np.float32)
    for p in range(W.shape[1]):
        m = np.where(W[:, p] > 0)[0]
        if len(m) == 0:
            continue
        X = sig[m] - sig[m].mean(1, keepdims=True)
        X = X / (X.std(1, keepdims=True) + 1e-12)
        U, Sv, Vt = np.linalg.svd(X, full_matrices=False)
        comp = Vt[0] * Sv[0]
        if np.corrcoef(comp, X.mean(0))[0, 1] < 0:
            comp = -comp
        out[p] = comp
    return out

def prepare(block, idx):
    """block is the whole (36000, 246, n_subj) array held in RAM, see the note below."""
    sig = block[:, :, idx].T[keep]
    sig = pca_merge(sig, W)
    sig = scipy.signal.resample_poly(sig, up=5, down=12, axis=1)   # 600 * 5/12 = 250
    return sig.T.astype(np.float32)                                 # (T, 100)

from ephys_tokenizer.models.ephys_tokenizer import EphysTokenizerModule
tok = EphysTokenizerModule.load_model(f'{MAMBA}/tokenizer/model')
print('tokeniser loaded')

tokeniser loaded


In [12]:
# Tokenise everyone.
#
# Two things matter for speed here.
#
# 1. The .mat files are chunked (1, 82, n_subj), so reading one subject at a time touches
#    every chunk and discards the other 76. That costs ~25 s per subject. Any slice along
#    the subject axis costs the same as the whole file, so we read in BATCHES: big enough
#    to amortise the read, small enough that memory stays flat. The full 2.7 GB block in
#    RAM pushed this machine into swap, which is why throughput collapsed on the first
#    attempt.
#
# 2. The tokeniser defaults to "cuda if available else cpu", so on Apple Silicon it runs
#    on CPU unless told otherwise. Passing device='mps' uses the GPU.
import gc
BATCH = 25          # subjects per read, ~0.9 GB each
DEVICE = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'tokenising on {DEVICE}, reading {BATCH} subjects at a time')

all_tok, all_lab, all_ids = [], [], []
t0 = time.time()
for path, label in [(f'{DATA}/ts48/tsAD.mat', 1), (f'{DATA}/ts48/tsCONT.mat', 0)]:
    with h5py.File(path, 'r') as f:
        n = f['matrix'].shape[2]
        ids = [''.join(chr(c) for c in np.asarray(f[r]).flatten())
               for r in f['radid'][:].flatten()]
        print(f'{os.path.basename(path)}: {n} subjects', flush=True)
        for lo in range(0, n, BATCH):
            hi = min(lo + BATCH, n)
            block = np.asarray(f['matrix'][:, :, lo:hi], dtype=np.float32)
            for j in range(hi - lo):
                all_tok.append(np.asarray(
                    tok.tokenize_session(prepare(block, j), device=DEVICE), np.uint8))
                all_ids.append(ids[lo + j])
                all_lab.append(label)
            del block; gc.collect()
            print(f'  {hi}/{n}   ({(time.time()-t0)/60:.1f} min)', flush=True)

tokens = np.stack(all_tok)
labels = np.array(all_lab, dtype=np.int64)
subj   = np.array(all_ids)

# The tokeniser's raw codebook has 128 entries but only 91 are used, and remap=True
# (its default) compacts them to 0..91 via label_map. The model has vocab_size 92, so
# anything above 91 would index past its embedding table.
VOCAB = 92
assert tokens.max() < VOCAB, (
    f'token id {tokens.max()} exceeds model vocab {VOCAB}. '
    'tokenize_session was probably called with remap=False.')

print(f'\ntokens {tokens.shape}  labels {labels.shape}  AD {int(labels.sum())}  control {int((labels==0).sum())}')
print(f'token range {tokens.min()} to {tokens.max()}, within vocab {VOCAB}')
print(f'total {(time.time()-t0)/60:.1f} min')

tokenising on mps, reading 25 subjects at a time
tsAD.mat: 77 subjects
  25/77   (0.9 min)
  50/77   (1.7 min)
  75/77   (2.5 min)
  77/77   (2.9 min)
tsCONT.mat: 84 subjects
  25/84   (3.8 min)
  50/84   (4.7 min)
  75/84   (20.5 min)
  84/84   (30.0 min)

tokens (161, 15000, 100)  labels (161,)  AD 77  control 84
token range 1 to 88, within vocab 92
total 30.0 min


In [13]:
# Session features, using MEG-Mamba's released transform so new recordings are processed
# exactly as the training corpus was.
from compute_session_features import featurize_session
tr = np.load(f'{MAMBA}/data/session_feature_transform.npz')
print('transform keys:', list(tr.keys()))

feats = np.stack([featurize_session(tokens[i], tr) for i in range(len(tokens))])
print(f'session features {feats.shape}   expect (161, 100, 292)')
assert feats.shape[2] == 292, f'feature dim {feats.shape[2]} does not match sess_emb_feat_dim 292'
print(f'finite {np.isfinite(feats).all()}   range {feats.min():.2f} to {feats.max():.2f}')

transform keys: ['vocab', 'n_parcels', 'bigram_pca', 'u_mu', 'u_sd', 'b_mu', 'b_sd', 'pca_components', 'pca_mean']
session features (161, 100, 292)   expect (161, 100, 292)
finite True   range -72.23 to 51.81


In [14]:
np.save(f'{OUT}/tokens_schaefer100.npy', tokens)
np.save(f'{OUT}/session_feats.npy', feats.astype(np.float32))
np.save(f'{OUT}/labels_mamba.npy', labels)
np.save(f'{OUT}/subject_ids_mamba.npy', subj)
print('saved to', OUT)
for f_ in ['tokens_schaefer100.npy', 'session_feats.npy']:
    mb = os.path.getsize(f'{OUT}/{f_}') / 1e6
    print(f'  {f_:28s} {mb:7.1f} MB')
print()
print('Next: upload tokens_schaefer100.npy and session_feats.npy to Colab,')
print('run colab_extract.ipynb there, then bring mamba_features.npy back here for notebook 03.')

saved to ../results_mamba
  tokens_schaefer100.npy         241.5 MB
  session_feats.npy               18.8 MB

Next: upload tokens_schaefer100.npy and session_feats.npy to Colab,
run colab_extract.ipynb there, then bring mamba_features.npy back here for notebook 03.
